In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Function to evaluate model
def evaluate_model(model, dataloader, device):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for (anchor, pos, neg) in tqdm(dataloader, desc="Evaluating"):
            anchor_img, anchor_feat = anchor
            neg_img, neg_feat = neg
            
            anchor_img, neg_img = anchor_img.to(device), neg_img.to(device)
            anchor_feat, neg_feat = anchor_feat.to(device), neg_feat.to(device)

            # Get embeddings
            anchor_out = model(anchor_img, anchor_feat)
            neg_out = model(neg_img, neg_feat)

            # Compute distances
            distances = F.pairwise_distance(anchor_out, neg_out)
            
            # Prediction: If distance is large, classify as forged (1), otherwise genuine (0)
            threshold = 0.5  # You might need to adjust this threshold based on performance
            preds = (distances > threshold).cpu().numpy()

            y_pred.extend(preds)
            y_true.extend([1] * len(preds))  # All negative pairs are forged (1)

    # Compute metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=1)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    conf_matrix = confusion_matrix(y_true, y_pred)

    # Print results
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")
    print("Confusion Matrix:")
    print(conf_matrix)

# Load the trained model
model.load_state_dict(torch.load("../Model/siamese_signature_model.pth", map_location=device))
model.to(device)

# Run evaluation
evaluate_model(model, val_loader, device)
